# 🏛️ فلسفة نظام RubikCare - النسخة العملية

## 🎯 المبادئ الأساسية

### 1. فصل النظام عن المؤسسات
- النظام يدير: 3 أدوار فقط (Admin, Doctor, Pharmacist)
- المؤسسات تدير: موظفيها وأدوارهم الداخلية

### 2. التدرج في الخدمات
- مستوى 0: مستخدم عادي (قائمة شخصية فقط)
- مستوى 1: صاحب مهنة (إنشاء مؤسسة غير موثقة)
- مستوى 2: مؤسسة موثقة (خدمات متقدمة)

### 3. الهوية قبل الدور
1. أنت إنسان (UserProfile)
2. تمارس مهنة (دور نظامي)
3. تعمل في مكان (عضوية في مؤسسة)
4. تؤدي وظيفة (دور داخل المؤسسة)

## 🗺️ تدفق المستخدمين

### للمهنيين (طبيب/صيدلي):
1. تسجيل ← طلب دور مهني ← تحقق من الترخيص
2. بعد الموافقة ← إنشاء مؤسسة (عيادة/صيدلية)
3. دعوة موظفين ← منح صلاحيات داخلية
4. طلب توثيق المؤسسة ← خدمات متقدمة

### للمستخدمين العاديين:
1. تسجيل ← حساب شخصي
2. الانضمام لمؤسسة موجودة
3. إنشاء مؤسسة غير طبية (شركة، معمل، etc.)

## ⚙️ النظام التقني المبني على الفلسفة

### الأدوار النظامية (3 فقط):
1. **Admin** - إدارة المنصة
2. **Doctor** - ممارسة الطب (يتطلب ترخيص)
3. **Pharmacist** - ممارسة الصيدلة (يتطلب ترخيص)

### المؤسسات:
- كل مؤسسة تدير موظفيها داخلياً
- الأدوار داخل المؤسسة: غير محدودة، غير مرتبطة بالنظام
- القوائم الجانبية: تظهر حسب العضوية في المؤسسة

## 🔄 سير العمل

### عند تسجيل دخول مستخدم:
1. هل له دور نظامي؟ ← نعم: عرض قوائم الدور
2. هل له عضوية في مؤسسات؟ ← نعم: عرض قوائم كل مؤسسة
3. هل هو مهني بدون مؤسسة؟ ← نعم: عرض زر "إنشاء مؤسسة"
4. الإفتراضي: القائمة الشخصية فقط

## القائمة الشخصية (الافتراضية عند الدخول)
├── 🏠 ملفي الشخصي
├── ⚙️ إعدادات عامة
├── 🔔 الإشعارات
├── 📋 طلباتي
│
└── 🔄 التبويبات المؤسسية (إذا كان له عضوية)
    ├── 🏢 [اسم المؤسسة 1] ← عند الضغط: تظهر قوائم المؤسسة فقط
    ├── 🏥 [اسم المؤسسة 2] ← عند الضغط: تظهر قوائم المؤسسة فقط  
    └── 🖥️ إدارة المنصة (إذا كان Admin)

# 📘 **الدليل الشامل لنظام الهوية والمصادقة - RubikCare v4**

## 📅 تاريخ الإصدار
**17 يناير 2026** - بعد حل مشكلة Migration وتثبيت نظام PSP

## 🎯 **ملخص النظام الحالي**

```
🏗️ هيكل الهوية المتكامل:
├── 👤 Microsoft Identity (جاهز)
│   ├── AspNetUsers (المستخدمون الأساسيون)
│   ├── AspNetRoles (الأدوار النظامية)
│   └── AspNetUserRoles (الربط بينهم) ⭐ معدل وموسع
│
├── 👥 UserProfiles (بيانات المستخدم الموسعة)
│   ├── UserProfileID (PK) - مرتبط بـ AspNetUsers
│   └── بيانات شخصية ومهنية
│
└── 🏢 Organizations & Memberships (النظام المؤسسي)
    ├── المنظمات (شركات، مستشفيات، صيدليات)
    └── العضويات (ربط الأشخاص بالمنظمات)
```

---

## 🔐 **الجزء 1: التصميم المعماري الحالي**

### **1.1 👤 جداول Microsoft Identity (الجاهزة)**

#### **AspNetUsers (المستخدمون الأساسيون):**
```sql
-- الهيكل القياسي + تعديلاتنا
Id (PK, nvarchar)               -- GUID
UserName, NormalizedUserName
Email, NormalizedEmail
PasswordHash, SecurityStamp
PhoneNumber, PhoneNumberConfirmed
TwoFactorEnabled, LockoutEnabled
AccessFailedCount, LockoutEnd
ConcurrencyStamp

-- ⭐ تعديلاتنا الإضافية:
UserProfileID (FK → UserProfiles)  -- رابط إلى البيانات الموسعة
DisplayName (nvarchar)             -- اسم العرض
LastActivityDate (datetime2)       -- آخر نشاط
```

#### **AspNetRoles (الأدوار النظامية):**
```sql
-- الهيكل القياسي
Id (PK, nvarchar)     -- GUID
Name (nvarchar)       -- "Admin", "Doctor", "Pharmacist"
NormalizedName        -- أحرف كبيرة
ConcurrencyStamp
```

#### **AspNetUserRoles (⭐ المعدل والموسع):**
```sql
-- ⭐ المفتاح الأساسي: مركب (UserId + RoleId)
UserId (PK, FK → AspNetUsers)
RoleId (PK, FK → AspNetRoles)

-- ⭐⭐ توسعاتنا لنظام الأدوار الوظيفية:
UserProfileID (FK → UserProfiles)  NOT NULL  -- أي شخص له هذا الدور
RoleType (nvarchar(50)) NOT NULL             -- "System", "Professional", "Organizational"
ProfessionalLicense (nvarchar(100)) NULL     -- ترخيص مهني
Specialization (nvarchar(200)) NULL          -- تخصص فرعي
IsVerified (bit) DEFAULT 0                   -- تم التحقق؟
VerifiedDate (datetime2) NULL
LicenseExpiryDate (datetime2) NULL

-- ⭐ إعدادات الوقت:
CreatedDate (datetime2) DEFAULT GETDATE()
ValidFrom (datetime2) NULL
ValidTo (datetime2) NULL
IsActive (bit) DEFAULT 1

-- ⭐ علاقات مع المؤسسات والمسميات:
OrganizationID (FK → Organizations) NULL     -- إذا كان الدور مرتبط بمؤسسة
SpecialityID (FK → Specialities) NULL        -- التخصص الطبي
SystemJobTitleID (FK → SystemJobTitles) NULL -- المسمى النظامي
CustomJobTitleID (FK → CustomJobTitles) NULL -- المسمى المخصص
```

### **1.2 👥 UserProfiles (بيانات المستخدم الموسعة)**

```sql
UserProfileID (PK, int)                -- معرف فريد
ApplicationUserId (FK → AspNetUsers)   -- رابط العكس
FirstName, LastName, FullNameAr, FullNameEn
NationalID, Email, PhoneNumber
DateOfBirth, Gender
ProfilePictureUrl
AreaID (FK → Areas)                    -- الموقع الجغرافي
IsActive, CreatedDate, LastModifiedDate
Address, Latitude, Longitude

-- ⭐ للترحيل:
OldProfileID, OldSupProfileID          -- ربط مع النظام القديم
```

### **1.3 🏢 النظام المؤسسي**

#### **Organizations (المنظمات):**
```sql
OrganizationID (PK, int)
OrganizationNameAr, OrganizationNameEn
OrganizationType (nvarchar)            -- "PHARMA", "HOSPITAL", "PHARMACY"
FounderUserProfileID (FK → UserProfiles)
AreaID (FK → Areas)
IsActive, CreatedDate
-- ... حقول أخرى
```

#### **OrgMemberships (العضويات):**
```sql
OrgMembershipID (PK, int)
UserProfileID (FK → UserProfiles)      -- الشخص
OrganizationID (FK → Organizations)    -- المنظمة
MembershipType (nvarchar)              -- "OWNER", "EMPLOYEE", "MEMBER"
SystemJobTitleID (FK → SystemJobTitles)
CustomJobTitleID (FK → CustomJobTitles)
IsActive, StartDate, EndDate
AssignedByUserID (FK → UserProfiles)   -- من قام بالتعيين
```

---

## ⚠️ **الجزء 2: المحاذير والتحذيرات الحرجة**

### **🔴 2.1 محاذير لا يمكن تخطيها:**

#### **❌ ممنوع تماماً:**
1. **لا تعدل `AspNetUserRoles` يدوياً في قاعدة البيانات**
2. **لا تحذف سجلات `AspNetUsers` بدون حذف `UserProfiles` أولاً**
3. **لا تستخدم `[Key]` في `UserRole.cs` - المفتاح مركب (UserId, RoleId)**
4. **لا تغير أنواع مفاتيح Identity (تبقى nvarchar(450))**

#### **⚠️ متى تفشل Migration:**
```text
عند حدوث أي من:
1. بيانات AspNetUserRoles تحتوي على UserProfileID غير موجود
2. محاولة تغيير نوع المفتاح الأساسي
3. تضارب في استدعاء base.OnModelCreating مرتين
4. علاقات دائرية مع DeleteBehavior خاطئ
```

### **🟡 2.2 إجراءات تتطلب حذراً:**

#### **إضافة مستخدم جديد:**
```csharp
// ✅ الصحيح (تسلسل آمن):
1. أنشئ UserProfile أولاً
2. ثم أنشئ ApplicationUser بربطه بـ UserProfileID
3. ثم أضف الأدوار في AspNetUserRoles

// ❌ الخطأ:
إنشاء AspNetUser بدون UserProfile
```

#### **حذف مستخدم:**
```csharp تحتاج مراجعة لإنه لا يجب حذف مستخدم من النظام ولكن ممكن soft delete
// ✅ الصحيح (تسلسل عكسي):
1. احذف من AspNetUserRoles (الأدوار)
2. احذف من OrgMemberships (العضويات)
3. احذف من UserRole (أدوار مخصصة إذا كانت)
4. احذف AspNetUser
5. احذف UserProfile

// ⚠️ تحذير: DeleteBehavior.Cascade على بعض العلاقات
```

### **🔵 2.3 أفضل الممارسات المختبرة:**

#### **للعلاقات:**
```csharp
// ✅ Restrict للأمان (يمنع الحذف إذا كان هناك مراجع)
.OnDelete(DeleteBehavior.Restrict)

// ✅ Cascade فقط للعلاقات التابعة حقاً
// مثال: UserRole مع UserProfile (الدور تابع للشخص)
.OnDelete(DeleteBehavior.Cascade)

// ❌ تجنب SetNull مع Identity
```

#### **لفهرسة AspNetUserRoles:**
```csharp
// ✅ لمنع تكرار الأدوار النشطة لنفس الشخص
.HasIndex(ur => new { ur.UserProfileID, ur.RoleType })
    .IsUnique()
    .HasFilter("[IsActive] = 1");

// ✅ للبحث السريع بالأدوار النشطة
.HasIndex(ur => new { ur.UserProfileID, ur.RoleType, ur.IsActive })
    .HasFilter("[IsActive] = 1");
```

---

## 🔄 **الجزء 3: سير العمل الموصى به**

### **3.1 📝 إنشاء مستخدم جديد (دليل خطوة بخطوة):**

#### **الخطوة 1: إنشاء UserProfile**
```csharp
var userProfile = new UserProfile
{
    FirstName = "أحمد",
    LastName = "محمد",
    PhoneNumber = "0123456789",
    Email = "ahmed@example.com",
    // ... باقي البيانات
};
await _context.UserProfiles.AddAsync(userProfile);
await _context.SaveChangesAsync();
```

#### **الخطوة 2: إنشاء ApplicationUser**
```csharp
var user = new ApplicationUser
{
    UserName = "ahmed@example.com",
    Email = "ahmed@example.com",
    UserProfileID = userProfile.UserProfileID, // ⭐ الرابط
    PhoneNumber = "0123456789"
};
var result = await _userManager.CreateAsync(user, "Password123!");
```

#### **الخطوة 3: إضافة الأدوار**
```csharp
// دور نظامي (Identity)
await _userManager.AddToRoleAsync(user, "Doctor");

// دور مخصص في UserRole (مع بيانات إضافية)
var userRole = new UserRole
{
    UserId = user.Id,
    RoleId = roleId, // من AspNetRoles
    UserProfileID = userProfile.UserProfileID,
    RoleType = "Professional",
    ProfessionalLicense = "MED-12345",
    Specialization = "جراحة القلب",
    IsActive = true
};
_context.UserRoles.Add(userRole);
await _context.SaveChangesAsync();
```

#### **الخطوة 4: إضافة عضوية مؤسسية (إذا لزم)**
```csharp
var membership = new OrgMembership
{
    UserProfileID = userProfile.UserProfileID,
    OrganizationID = hospitalId,
    MembershipType = "DOCTOR",
    SystemJobTitleID = doctorTitleId,
    IsActive = true,
    StartDate = DateTime.Now
};
```

### **3.2 🔍 استعلام بيانات المستخدم:**

#### **للحصول على بيانات كاملة:**
```csharp
var userWithDetails = await _context.UserProfiles
    .Include(up => up.ApplicationUser)
    .Include(up => up.UserRoles)
        .ThenInclude(ur => ur.Role)
    .Include(up => up.OrgMemberships)
        .ThenInclude(om => om.Organization)
    .Include(up => up.Area)
        .ThenInclude(a => a.City)
        .ThenInclude(c => c.Country)
    .FirstOrDefaultAsync(up => up.UserProfileID == id);
```

#### **للتحقق من الصلاحيات:**
```csharp
public bool HasRole(int userProfileId, string roleType)
{
    return _context.UserRoles
        .Any(ur => ur.UserProfileID == userProfileId && 
                   ur.RoleType == roleType && 
                   ur.IsActive && 
                   (ur.ValidTo == null || ur.ValidTo > DateTime.Now));
}
```

---

## 🛡️ **الجزء 4: الأمان والصلاحيات**

### **4.1 🔐 مستويات الصلاحيات:**

```
مستوى الصلاحيات (من الأعلى للأدنى):
1. 🔴 النظامية (System): إدارة المنصة كاملة
2. 🟢 المهنية (Professional): ممارسة المهنة (طبيب، صيدلي)
3. 🟡 المؤسسية (Organizational): داخل مؤسسة معينة
4. 🔵 العضوية (Membership): صلاحيات محدودة في مؤسسة
```

### **4.2 ✅ التحقق من الصلاحيات:**

```csharp
public class AuthorizationService
{
    // 1. هل لديه دور نظامي؟
    public async Task<bool> IsInSystemRoleAsync(string userId, string role)
    {
        return await _userManager.IsInRoleAsync(userId, role);
    }
    
    // 2. هل لديه دور مهني نشط؟
    public bool HasProfessionalRole(int userProfileId, string roleType)
    {
        return _context.UserRoles.Any(ur =>
            ur.UserProfileID == userProfileId &&
            ur.RoleType == roleType &&
            ur.IsActive &&
            ur.ValidTo > DateTime.Now);
    }
    
    // 3. هل هو عضو نشط في مؤسسة؟
    public bool IsActiveMember(int userProfileId, int organizationId)
    {
        return _context.OrgMemberships.Any(om =>
            om.UserProfileID == userProfileId &&
            om.OrganizationID == organizationId &&
            om.IsActive &&
            om.EndDate > DateTime.Now);
    }
}
```

### **4.3 ⚠️ ثغرات محتملة وتحذيرات:**

#### **ثغرة الوقت:**
```csharp
// ❌ خطأ: عدم التحقق من صلاحية الدور الزمنية
var hasRole = userRoles.Any(ur => ur.IsActive);

// ✅ صحيح: التحقق من التاريخ
var hasValidRole = userRoles.Any(ur => 
    ur.IsActive && 
    ur.ValidFrom <= DateTime.Now && 
    (ur.ValidTo == null || ur.ValidTo > DateTime.Now));
```

#### **ثغرة الترخيص:**
```csharp
// ❌ خطأ: السماح لمزاولة المهنة بدون ترخيص ساري
if (userRole.RoleType == "Doctor") { /* ممارسة */ }

// ✅ صحيح: التحقق من الترخيص
if (userRole.RoleType == "Doctor" && 
    userRole.IsVerified && 
    userRole.LicenseExpiryDate > DateTime.Now)
{
    /* ممارسة */
}
```

---

## 🔧 **الجزء 5: الصيانة والاستكشاف**

### **5.1 🩹 إصلاح مشاكل شائعة:**

#### **مشكلة: UserProfileID غير متطابق**
```sql
-- التشخيص:
SELECT u.Id, u.UserProfileID, up.UserProfileID
FROM AspNetUsers u
LEFT JOIN UserProfiles up ON u.UserProfileID = up.UserProfileID
WHERE up.UserProfileID IS NULL;

-- الإصلاح:
UPDATE u
SET u.UserProfileID = up.UserProfileID
FROM AspNetUsers u
INNER JOIN UserProfiles up ON u.Email = up.Email  -- أو أي معيار مطابقة
WHERE u.UserProfileID IS NULL;
```

#### **مشكلة: أدوار مكررة نشطة**
```sql
-- البحث عن تكرار:
SELECT UserProfileID, RoleType, COUNT(*) as DuplicateCount
FROM AspNetUserRoles
WHERE IsActive = 1
GROUP BY UserProfileID, RoleType
HAVING COUNT(*) > 1;

-- إصلاح (تعطيل المكررات):
WITH Duplicates AS (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY UserProfileID, RoleType ORDER BY CreatedDate DESC) as RowNum
    FROM AspNetUserRoles
    WHERE IsActive = 1
)
UPDATE Duplicates
SET IsActive = 0
WHERE RowNum > 1;
```

### **5.2 📊 مراقبة النظام:**

#### **استعلامات المراقبة المهمة:**
```sql
-- 1. المستخدمون بدون UserProfile
SELECT COUNT(*) FROM AspNetUsers WHERE UserProfileID IS NULL;

-- 2. الأدوار منتهية الصلاحية لا تزال نشطة
SELECT COUNT(*) FROM AspNetUserRoles 
WHERE IsActive = 1 AND ValidTo < GETDATE();

-- 3. تراخيص منتهية
SELECT COUNT(*) FROM AspNetUserRoles 
WHERE LicenseExpiryDate < GETDATE() AND IsActive = 1;

-- 4. عضوايت منتهية لا تزال نشطة
SELECT COUNT(*) FROM OrgMemberships 
WHERE IsActive = 1 AND EndDate < GETDATE();
```

#### **تقرير الصحة الشهري:**
```sql
SELECT 
    'Users' AS Category,
    (SELECT COUNT(*) FROM AspNetUsers) AS Total,
    (SELECT COUNT(*) FROM AspNetUsers WHERE UserProfileID IS NULL) AS Issues
UNION ALL
SELECT 
    'Active Roles',
    (SELECT COUNT(*) FROM AspNetUserRoles WHERE IsActive = 1),
    (SELECT COUNT(*) FROM AspNetUserRoles WHERE IsActive = 1 AND ValidTo < GETDATE())
UNION ALL
SELECT 
    'Licenses',
    (SELECT COUNT(*) FROM AspNetUserRoles WHERE ProfessionalLicense IS NOT NULL),
    (SELECT COUNT(*) FROM AspNetUserRoles WHERE LicenseExpiryDate < GETDATE());
```

---

## 🎯 **الجزء 6: التكامل مع PSP والأنظمة الأخرى**

### **6.1 🔗 ربط PSP بنظام الهوية:**

#### **PSP Patient مع UserProfile:**
```csharp
// في PSPPatient
public int PatientProfileID { get; set; }  // FK → UserProfiles
public virtual UserProfile PatientProfile { get; set; }

// في PSP Enrollment
public int DoctorID { get; set; }  // FK → UserProfiles (كطبيب)
public virtual UserProfile Doctor { get; set; }
```

#### **التحقق من صلاحية وصف PSP:**
```csharp
public bool CanPrescribePSP(int doctorProfileId, int programId)
{
    // 1. التحقق من الدور المهني
    var hasDoctorRole = _context.UserRoles.Any(ur =>
        ur.UserProfileID == doctorProfileId &&
        ur.RoleType == "Doctor" &&
        ur.IsActive &&
        ur.IsVerified);
    
    // 2. التحقق من العضوية في البرنامج
    var isEnrolled = _context.PSPEnrollments.Any(e =>
        e.DoctorID == doctorProfileId &&
        e.ProgramID == programId &&
        e.Status == "APPROVED" &&
        e.IsActive);
    
    return hasDoctorRole && isEnrolled;
}
```

### **6.2 ⚡ نصائح للأداء:**

#### **فهارس ضرورية:**
```sql
-- 1. للبحث السريع بالأدوار النشطة
CREATE INDEX IX_UserRoles_Active 
ON AspNetUserRoles(UserProfileID, RoleType) 
WHERE IsActive = 1;

-- 2. للتحقق من الصلاحية الزمنية
CREATE INDEX IX_UserRoles_Validity 
ON AspNetUserRoles(ValidTo) 
WHERE IsActive = 1 AND ValidTo IS NOT NULL;

-- 3. للربط مع UserProfiles
CREATE INDEX IX_AspNetUsers_UserProfileID 
ON AspNetUsers(UserProfileID) 
WHERE UserProfileID IS NOT NULL;
```

#### **تحسين استعلامات الهوية:**
```csharp
// ❌ بطيء: تحميل كل شيء
var user = await _context.UserProfiles
    .Include(up => up.ApplicationUser)
    .Include(up => up.UserRoles)
    .Include(up => up.OrgMemberships)
    .FirstOrDefaultAsync(up => up.UserProfileID == id);

// ✅ سريع: تحميل حسب الحاجة
var user = await _context.UserProfiles
    .Select(up => new
    {
        up.UserProfileID,
        up.FullNameAr,
        up.Email,
        up.PhoneNumber,
        Roles = up.UserRoles.Where(ur => ur.IsActive).Select(ur => ur.RoleType),
        ActiveMemberships = up.OrgMemberships.Where(om => om.IsActive).Count()
    })
    .FirstOrDefaultAsync(up => up.UserProfileID == id);
```

---

## 📞 **الجزء 7: جهات الاتصال والمراجع**

### **7.1 📁 ملفات النظام الرئيسية:**

```
📁 Data/Models/Identity/
├── ApplicationUser.cs        // AspNetUsers الموسع
├── ApplicationRole.cs        // AspNetRoles الموسع (إذا احتاج)
├── UserRole.cs              // ⭐ المعدل (بدون UserRoleID)
└── UserProfile.cs           // البيانات الموسعة

📁 Data/Models/
├── Organization.cs          // المنظمات
├── OrgMembership.cs         // العضويات
└── SystemJobTitle.cs        // المسميات النظامية

📁 Data/
└── BusinessDbContext.cs     // ⭐ منطقة ConfigureIdentityAndMemberships
```

### **7.2 🔗 وثائق مرجعية:**

1. **الدليل التقني:** `TECHNICAL-IMPLEMENTATION-GUIDE.md`
2. **لوحة التحكم:** `Project-Dashboard.md`
3. **خطة PSP:** `PSP-plan.md`
4. **سجل المشاكل:** `Migration-Fix-Log.txt`

### **7.3 ⏰ جدول الصيانة الدورية:**

| المهمة | التكرار | الوصف |
|--------|---------|--------|
| **فحص البيانات** | أسبوعياً | التحقق من تكامل العلاقات |
| **مراجعة الصلاحيات** | شهرياً | تعطيل الأدوار المنتهية |
| **تحديث التراخيص** | حسب الصلاحية | تنبيهات انتهاء الترخيص |
| **تقرير الصحة** | ربع سنوي | تقرير شامل عن النظام |

---

## 📘 **الجزء 8: التنفيذ العملي في صفحات Blazor - محدث كاملاً**

### 🔧 **8.1 القاعدة الذهبية للهوية في RubikCare**

#### **⚖️ الفهم الأساسي:**
```csharp
// ✅ **النماذج الموسعة (Extended Models):**
// - موجودة في Data/Models/
// - لها DbSet في BusinessDbContext
// - تستخدم IGenericService<>
// أمثلة: UserRole, UserProfile, Organization, Area

// ✅ **النماذج الأساسية (Core Identity Models):**
// - جزء من Microsoft.AspNetCore.Identity
// - ليست في DbContext
// - تستخدم RoleManager/UserManager
// أمثلة: IdentityRole, ApplicationUser

// ❌ **الخطأ الفادح:**
// محاولة استخدام IGenericService<IdentityRole> أو IGenericService<ApplicationUser>
```

---

### 🛠️ **8.2 مثال عملي: صفحة إدارة الأدوار (`/admin/users/roles`)**

#### **✅ الكود الصحيح:**
```razor
@page "/admin/users/roles"
@using Microsoft.AspNetCore.Identity
@using Rubikcare.Web.Data.Models

@inherits BaseCrudPage<UserRole>  // ⭐ النموذج الموسع

@* ⭐ الخدمات الصحيحة: *@
@inject RoleManager<IdentityRole> RoleManager          // ⚠️ للـ AspNetRoles (مايكروسوفت)
@inject IGenericService<UserRole> UserRoleService      // ✅ للـ AspNetUserRoles (الموسع)
@inject IGenericService<UserProfile> UserProfileService // ✅ للنماذج الموسعة

@code {
    // ⭐ القوائم المرجعية:
    private List<IdentityRole> allSystemRoles = new();   // من RoleManager
    private List<UserProfile> allUserProfiles = new();   // من IGenericService
    
    protected override async Task QueryDataAsync()
    {
        try
        {
            // ⭐ 1. جلب الأدوار النظامية (من Microsoft.Identity)
            allSystemRoles = RoleManager.Roles.ToList();
            
            // ⭐ 2. جلب الملفات الشخصية (من النماذج الموسعة)
            allUserProfiles = (await UserProfileService.GetAllAsync()).ToList();
            
            // ⭐ 3. جلب تعيينات الأدوار (من BaseCrudPage)
            await LoadAllItemsAsync();
        }
        catch (Exception ex)
        {
            // معالجة الخطأ
        }
    }
}
```

---

### 📋 **8.3 جدول المراجعة السريع (Quick Reference Table)**

| النموذج | المسار في DB | الموقع | Service الصحيح | مثال الاستخدام |
|---------|-------------|--------|----------------|----------------|
| **UserRole** | `AspNetUserRoles` | `Data/Models/Identity/` | `IGenericService<UserRole>` | إدارة تعيينات الأدوار |
| **IdentityRole** | `AspNetRoles` | `Microsoft.Identity` | `RoleManager<IdentityRole>` | عرض قائمة الأدوار |
| **UserProfile** | `UserProfiles` | `Data/Models/` | `IGenericService<UserProfile>` | إدارة الملفات الشخصية |
| **ApplicationUser** | `AspNetUsers` | `Microsoft.Identity` | `UserManager<ApplicationUser>` | إدارة حسابات النظام |
| **Organization** | `Organizations` | `Data/Models/` | `IGenericService<Organization>` | إدارة المؤسسات |
| **Area** | `Areas` | `Data/Models/` | `IGenericService<Area>` | إدارة المناطق |

---

### ⚠️ **8.4 الأخطاء الشائعة وحلولها**

#### **المشكلة 1: `Cannot create a DbSet for 'IdentityRole<string>'`**
```csharp
// ❌ الخطأ:
@inject IGenericService<IdentityRole> RoleService

// ✅ الحل:
@inject RoleManager<IdentityRole> RoleManager
```

#### **المشكلة 2: `No registered service of type 'RoleManager<IdentityRole>'`**
```csharp
// ✅ التأكد من وجود في Program.cs:
builder.Services.AddIdentity<ApplicationUser, IdentityRole>()
    .AddRoleManager<RoleManager<IdentityRole>>()    // ⭐ مهم
    .AddEntityFrameworkStores<BusinessDbContext>();
```

#### **المشكلة 3: استخدام النموذج الخطأ في العلاقات**
```csharp
// ❌ خطأ: محاولة ربط IdentityRole مباشرة
public class UserRole : IdentityUserRole<string>
{
    // ⚠️ هذا صحيح لأن UserRole يرث من IdentityUserRole
}

// ❌ خطأ آخر: 
public virtual IdentityRole Role { get; set; } // ⚠️ يجب استخدام النموذج الموسع إن وجد
```

---

### 🔄 **8.5 نمط العمل الموصى به للصفحات الجديدة**

#### **الخطوة 1: تحديد نوع النموذج**
```csharp
public bool ShouldUseGenericService(Type modelType)
{
    // النماذج في Data/Models/ تستخدم IGenericService
    // النماذج في Microsoft.Identity تستخدم Managers
    
    var identityTypes = new List<Type>
    {
        typeof(IdentityRole),
        typeof(ApplicationUser)
    };
    
    return !identityTypes.Contains(modelType);
}
```

#### **الخطوة 2: استخدام النمط الموحد**
```razor
@* نموذج صفحة CRUD قياسية: *@
@inherits BaseCrudPage<Model>  // Model من Data/Models/

@if (Model is IdentityRole || Model is ApplicationUser)
{
    @* استخدام Manager *@
    @inject RoleManager<IdentityRole> RoleManager
}
else
{
    @* استخدام IGenericService *@
    @inject IGenericService<Model> DataService
}
```

---

### 🎯 **8.6 المشكلة الحالية: القوائم المنسدلة في صفحة المؤسسات**

#### **التحليل:**
```csharp
// المشكلة: القوائم المنسدلة (الدولة ← المدينة ← المنطقة) لا تعمل

// ✅ الحل المقترح للجلسة القادمة:
1. التحقق من أحداث التغيير (onchange) في عناصر select
2. التأكد من تحديث StateHasChanged() عند تغيير البيانات
3. فحص علاقة Area → City (استخدام AreaCityID بدلاً من CityID)
4. استخدام Console.WriteLine للتصحيح

// ⭐ كود التصحيح:
private void OnCountryChanged()
{
    Console.WriteLine($"Country changed to: {selectedCountryId}");
    Console.WriteLine($"Available cities: {allCities.Count(c => c.CountryID == selectedCountryId)}");
    
    // تحديث القوائم
    StateHasChanged();
}
```

---

### 📞 **8.7 مرجع سريع للتسجيلات في Program.cs**

```csharp
// ✅ التسجيلات الصحيحة للهوية:
builder.Services.AddIdentity<ApplicationUser, IdentityRole>(options =>
{
    // إعدادات الهوية
})
.AddEntityFrameworkStores<BusinessDbContext>()
.AddDefaultTokenProviders()
.AddRoleManager<RoleManager<IdentityRole>>()    // ⭐ مهم لصفحات الأدوار
.AddUserManager<UserManager<ApplicationUser>>(); // ⭐ مهم لصفحات المستخدمين
```

---

### 🚀 **8.8 أفضل الممارسات المختبرة**

#### **✅ افعل:**
1. **افصل الخدمات:** استخدم `IGenericService` للنماذج الموسعة فقط
2. **استخدم Managers:** للنماذج الأساسية (IdentityRole, ApplicationUser)
3. **تحقق أولاً:** قبل إنشاء صفحة، تحقق من نوع النموذج
4. **انسخ من الأمثلة:** استخدم `Cities.razor` للصفحات العادية

#### **❌ لا تفعل:**
1. **لا تفترض:** أن كل نموذج في DB يعمل مع `IGenericService`
2. **لا تخلط:** بين `@bind` و `@onchange` (استخدم `@bind:event="onchange"`)
3. **لا تعدل Migration:** اترك EF Core يعمل لوحده

---

### 📊 **8.9 قائمة التحقق قبل إنشاء صفحة جديدة**

```yaml
✅ التحقق 1: هل النموذج في Data/Models/ أم من Microsoft.Identity?
✅ التحقق 2: هل له DbSet في BusinessDbContext?
✅ التحقق 3: إذا كان من Microsoft.Identity، ما هو الـ Manager المناسب?
✅ التحقق 4: هل الـ Manager مسجل في Program.cs?
✅ التحقق 5: هل هناك مثال مشابه يعمل (مثل Cities.razor)?
```

---


## 🔍 **الجزء 9: استكشاف الأخطاء وإصلاحها**

### **9.1 🚨 رسائل الخطأ الشائعة وحلولها**

#### **المشكلة 1: `Cannot create a DbSet for 'IdentityRole<string>'`**
**السبب:** محاولة استخدام `IGenericService<IdentityRole>` بينما `IdentityRole` ليس في `DbContext`
**الحل:** استخدم `RoleManager<IdentityRole>` بدلاً من `IGenericService`

#### **المشكلة 2: `No registered service of type 'RoleManager<IdentityRole>'`**
**السبب:** `RoleManager` غير مسجل في `Program.cs`
**الحل:** تأكد من وجود:
```csharp
builder.Services.AddIdentity<ApplicationUser, IdentityRole>()
    .AddRoleManager<RoleManager<IdentityRole>>()
    .AddEntityFrameworkStores<BusinessDbContext>();
```

#### **المشكلة 3: `CS0305: Using the generic type 'RoleManager<TRole>' requires 1 type arguments`**
**السبب:** استخدام `RoleManager` بدون تحديد النوع
**الحل:** استخدم `RoleManager<IdentityRole>` بشكل كامل

### **9.2 📝 قائمة التحقق قبل إنشاء صفحة هوية جديدة**
```yaml
✅ التحقق 1: هل النموذج في Data/Models/ أم من Microsoft.Identity?
✅ التحقق 2: هل له DbSet في BusinessDbContext?
✅ التحقق 3: إذا كان من Microsoft.Identity، ما هو الـ Manager المناسب?
✅ التحقق 4: هل الـ Manager مسجل في Program.cs?
```

### **9.3 🔧 إصلاح سريع لأخطاء الهوية**
```csharp
// عند ظهور خطأ في صفحة هوية:
public async Task QuickFixIdentityPage()
{
    // 1. تحقق من @inject
    // 2. استبدل IGenericService<IdentityModel> بـ Manager المناسب
    // 3. تأكد من أن Manager مسجل في Program.cs
    // 4. استخدم Manager.Roles أو Manager.Users بدلاً من GetAllAsync()
}
```

### **9.4 📞 مرجع سريع للتسجيلات في Program.cs**
```csharp
// ✅ التسجيلات الصحيحة للهوية:
builder.Services.AddIdentity<ApplicationUser, IdentityRole>(options =>
{
    // إعدادات الهوية
})
.AddEntityFrameworkStores<BusinessDbContext>()
.AddDefaultTokenProviders()
.AddRoleManager<RoleManager<IdentityRole>>()    // ⭐ مهم لصفحات الأدوار
.AddUserManager<UserManager<ApplicationUser>>(); // ⭐ مهم لصفحات المستخدمين
```

### **9.5 🎯 أفضل الممارسات المختبرة**
1. **فصل الخدمات:** استخدم `IGenericService` للنماذج الموسعة فقط
2. **استخدام Managers:** للنماذج الأساسية (IdentityRole, ApplicationUser)
3. **التحقق المسبق:** قبل إنشاء صفحة، تحقق من نوع النموذج
4. **النسخ من الأمثلة:** استخدم `Cities.razor` للصفحات العادية، `UserRoles.razor` للهوية


## 🏁 **الخلاصة النهائية**

### **✅ مبادئ النظام الحالي:**

1. **فصل واضح:** هوية Microsoft + بيانات موسعة + نظام مؤسسي
2. **مرونة:** أدوار متعددة المستويات (نظامية، مهنية، مؤسسية)
3. **أمان:** Restrict للعلاقات + صلاحيات زمنية
4. **تتبع:** تاريخ الإنشاء، التعديل، الصلاحية

### **⚠️ التذكيرات الدائمة:**

1. **لا تلمس Migration** - دعه يعمل لوحده
2. **UserRoleID محذوف** - المفتاح مركب (UserId, RoleId)
3. **base.OnModelCreating مرة واحدة** - في الطريقة الرئيسية فقط
4. **تسلسل العمليات مهم** - إنشاء وحذف بالترتيب الصحيح

### **🚀 المسار المقبل:**

1. **تطوير واجهات PSP** باستخدام BaseCrudPage
2. **تكامل PSP مع نظام الهوية**
3. **إضافة نظام الإشعارات** للصلاحيات والتراخيص
4. **تحسينات الأداء** لفهارس الاستعلامات الشائعة

---

**📅 آخر تحديث:** 17 يناير 2026  
**✅ الحالة:** نظام مستقر - Migration نظيفة  
**🎯 المرحلة التالية:** تطوير واجهات PSP  

**ممتاز!** دعنا **نرتب الأولويات ونحدد الصفحات المطلوبة**:

---

## 🗺️ **خريطة الصفحات المطلوبة لنظام الهوية والمؤسسات**

### **📁 1. صفحات إدارة المنصة (Admin)**

```
/admin/identity/
├── /users/list          ← إدارة المستخدمين (جاهزة)
├── /users/roles         ⭐ **جديد**: إدارة أدوار المستخدمين
├── /users/{id}/edit     ← تعديل مستخدم (جاهزة)
└── /organizations/list  ← قائمة المنظمات (جاهزة)
```

### **📁 2. صفحات إدارة المؤسسات (Organization Management)**

```
/organization/
├── /create              ⭐ **جديد**: إنشاء مؤسسة جديدة
├── /{id}/dashboard      ⭐ **جديد**: لوحة تحكم المؤسسة
├── /{id}/members        ⭐ **جديد**: إدارة الأعضاء
├── /{id}/settings       ⭐ **جديد**: إعدادات المؤسسة
├── /{id}/services       ⭐ **جديد**: خدمات المؤسسة
└── /{id}/psp           ⭐ **جديد**: برامج دعم المرضى (لشركات الأدوية)
```

### **📁 3. صفحات المستخدم الشخصية (User Portal)**

```
/profile/
├── /my-profile          ← ملفي الشخصي (جاهزة)
├── /my-organizations    ⭐ **جديد**: مؤسساتي
├── /my-roles            ⭐ **جديد**: أدواري النظامية
└── /invitations         ⭐ **جديد**: دعوات الانضمام
```

---

## 🎯 **ترتيب الأولويات (من الأهم للأقل):**

### **🔥 المرحلة 1: أساسيات النظام (أسبوع واحد)**

| # | الصفحة | المسار | الوصف | الأولوية |
|---|--------|--------|--------|-----------|
| 1 | **إدارة أدوار المستخدمين** | `/admin/users/roles` | إضافة/حذف أدوار للمستخدمين | ⭐⭐⭐⭐⭐ |
| 2 | **إنشاء مؤسسة جديدة** | `/organization/create` | للمهنيين (أطباء/صيادلة) | ⭐⭐⭐⭐⭐ |
| 3 | **لوحة تحكم المؤسسة** | `/organization/{id}/dashboard` | صفحة رئيسية لكل مؤسسة | ⭐⭐⭐⭐⭐ |

### **⚡ المرحلة 2: إدارة المؤسسات (أسبوع واحد)**

| # | الصفحة | المسار | الوصف | الأولوية |
|---|--------|--------|--------|-----------|
| 4 | **إدارة أعضاء المؤسسة** | `/organization/{id}/members` | دعوة وإدارة الموظفين | ⭐⭐⭐⭐ |
| 5 | **إعدادات المؤسسة** | `/organization/{id}/settings` | بيانات المؤسسة الأساسية | ⭐⭐⭐⭐ |
| 6 | **مؤسساتي** | `/profile/my-organizations` | قائمة مؤسسات المستخدم | ⭐⭐⭐ |

### **📅 المرحلة 3: تحسينات متقدمة (أسبوعين)**

| # | الصفحة | المسار | الوصف | الأولوية |
|---|--------|--------|--------|-----------|
| 7 | **برامج PSP للمؤسسات** | `/organization/{id}/psp` | لشركات الأدوية | ⭐⭐⭐ |
| 8 | **خدمات المؤسسة** | `/organization/{id}/services` | المنتجات والخدمات | ⭐⭐ |
| 9 | **نظام الدعوات** | `/profile/invitations` | قبول/رفض دعوات الانضمام | ⭐⭐ |

---

## 🏗️ **التفاصيل التقنية لكل صفحة:**

### **1. صفحة إدارة أدوار المستخدمين (`/admin/users/roles`)**
```razor
@page "/admin/users/roles"
@inherits BaseCrudPage<UserRoleViewModel>
```
**الوظائف:**
- عرض جميع المستخدمين مع أدوارهم
- إضافة دور جديد (Admin, Doctor, Pharmacist)
- تحديد RoleType (Platform/Professional/Organization)
- ربط بالمؤسسة إذا كان RoleType = Organization

### **2. صفحة إنشاء مؤسسة جديدة (`/organization/create`)**
```razor
@page "/organization/create"
```
**الشروط:**
- فقط للمستخدمين الذين لديهم دور **Doctor أو Pharmacist**
- يختار نوع المؤسسة (عيادة/صيدلية/شركة)
- يملأ البيانات الأساسية
- يصبح **مالك (OWNER)** للمؤسسة الجديدة

### **3. صفحة إدارة أعضاء المؤسسة (`/organization/{id}/members`)**
```razor
@page "/organization/{id}/members"
```
**الصلاحيات:**
- فقط الأعضاء الذين لديهم `CanManageMembers = true`
- دعوة أعضاء جدد (بالبريد الإلكتروني)
- تعديل صلاحيات الأعضاء الحاليين
- تعيين JobTitle لكل عضو

---

## 🔄 **سير العمل الكامل (بدون SQL يدوي):**

### **لطبيب جديد يريد الانضمام:**
```
1. تسجيل حساب جديد ← صفحة التسجيل
2. طلب دور "Doctor" ← مع رفع الرخصة
3. بعد موافقة الإدارة ← منح دور Doctor
4. الدخول للبوابة ← يرى زر "إنشاء مؤسسة جديدة"
5. إنشاء عيادة ← يصبح مالكاً لها
6. دعوة موظفين ← من صفحة إدارة الأعضاء
```

### **لمدير مؤسسة موجودة:**
```
1. تسجيل دخول ← يرى تبويبات مؤسساته
2. اختيار مؤسسة ← لوحة تحكم المؤسسة
3. إدارة الأعضاء ← دعوة موظفين جدد
4. إعدادات المؤسسة ← تحديث البيانات
5. إذا كانت شركة أدوية ← إضافة برامج PSP
```

---

## 📋 **الخطوات الفورية القادمة:**

### **اليوم/غداً:** بدء صفحة **إدارة أدوار المستخدمين**
- لأنها الأهم لاستكمال النظام
- تمكننا من إدارة كل شيء بدون SQL

### **بعدها:** صفحة **إنشاء مؤسسة جديدة**
- تكمل دورة حياة المستخدم المهني
- تحل مشكلة "طبيب بدون مؤسسة"

### **ثم:** صفحة **إدارة أعضاء المؤسسة**
- تكمل النظام المؤسسي
- تمكن المدراء من إدارة فرقهم

---



Rubikcare.Web/Components/Account/Pages/Manage/ChangePassword.razor
Rubikcare.Web\Components\Account\Pages\Manage\DeletePersonalData.razor
Rubikcare.Web\Components\Account\Pages\Manage\Disable2fa.razor
Rubikcare.Web\Components\Account\Pages\Manage\Email.razor
Rubikcare.Web\Components\Account\Pages\Manage\EnableAuthenticator.razor
Rubikcare.Web\Components\Account\Pages\Manage\ExternalLogins.razor
Rubikcare.Web\Components\Account\Pages\Manage\GenerateRecoveryCodes.razor
Rubikcare.Web\Components\Account\Pages\Manage\Index.razor
Rubikcare.Web\Components\Account\Pages\Manage\Passkeys.razor
Rubikcare.Web\Components\Account\Pages\Manage\PersonalData.razor
Rubikcare.Web\Components\Account\Pages\Manage\RenamePasskey.razor
Rubikcare.Web\Components\Account\Pages\Manage\ResetAuthenticator.razor
Rubikcare.Web\Components\Account\Pages\Manage\SetPassword.razor
Rubikcare.Web\Components\Account\Pages\Manage\TwoFactorAuthentication.razor
Rubikcare.Web\Components\Account\Pages\AccessDenied.razor
Rubikcare.Web\Components\Account\Pages\ConfirmEmail.razor
Rubikcare.Web\Components\Account\Pages\ConfirmEmailChange.razor
Rubikcare.Web\Components\Account\Pages\ExternalLogin.razor 
Rubikcare.Web\Components\Account\Pages\ForgotPassword.razor   مستخدم
Rubikcare.Web\Components\Account\Pages\ForgotPasswordConfirmation.razor مستخدم
Rubikcare.Web\Components\Account\Pages\InvalidPasswordReset.razor
Rubikcare.Web\Components\Account\Pages\InvalidUser.razor
CRubikcare.Web\Components\Account\Pages\Lockout.razor
Rubikcare.Web\Components\Account\Pages\Login.razor  مستخدم
Rubikcare.Web\Components\Account\Pages\LoginWith2fa.razor
Rubikcare.Web\Components\Account\Pages\LoginWithRecoveryCode.razor
Rubikcare.Web\Components\Account\Pages\Register.razor
Rubikcare.Web\Components\Account\Pages\RegisterConfirmation.razor
Rubikcare.Web\Components\Account\Pages\ResendEmailConfirmation.razor
Rubikcare.Web\Components\Account\Pages\ResendEmailConfirmation.razor
Rubikcare.Web\Components\Account\Pages\ResetPassword.razor
Rubikcare.Web\Components\Account\Pages\ResetPasswordConfirmation.razor
Rubikcare.Web\Components\Account\IdentityComponentsEndpointRouteBuilderExtensions.cs
Rubikcare.Web\Components\Account\IdentityNoOpEmailSender.cs
Rubikcare.Web\Components\Account\IdentityRedirectManager.cs
Rubikcare.Web\Components\Account\IdentityRevalidatingAuthenticationStateProvider.cs
Rubikcare.Web\Components\Account\PasskeyInputModel.cs
Rubikcare.Web\Components\Account\PasskeyOperation.cs
Rubikcare.Web/Components/Account/Shared/ExternalLoginPicker.razor
Rubikcare.Web\Components\Account\Shared\ManageLayout.razor
Rubikcare.Web\Components\Account\Shared\ManageNavMenu.razor
Rubikcare.Web\Components\Account\Shared\PasskeySubmit.razor
Rubikcare.Web\Components\Account\Shared\RedirectToLogin.razor
Rubikcare.Web\Components\Account\Shared\ShowRecoveryCodes.razor
Rubikcare.Web\Components\Account\Shared\StatusMessage.razor
ForgotPassword.razor مستخدم
logout.razor غير موجود في المشروع أصلا 
ربما تقصد lockout.razor 
ExternalLogin.razor لم يتم استخدامه
ResetPassword.razor مستخدم
ResetPasswordconfermation.razor
Index.razor غير مستخدم
ChangePassword.razor مستخدم
DeletePersonalData.razor غير مستخدم
PersonalData.razor غير مستخدم
Email.razorغير مستخدم
StatusMessage.razor غير مستخدم
ShowRecoveryCodes.razor غير مستخدم
RedirectToLogin.razor غير مستخدم 